# Bangladesh Seasonal Rice Mapping: National Scale Pipeline

This notebook executes the national Sentinel-1 seasonal rice classification pipeline across Bangladesh (408 tiles × 3 years: 2017, 2022, 2023).

**Key Specifications:**
- **Data**: Sentinel-1 RTC via Microsoft Planetary Computer (anonymous SAS signing, 0 cost)
- **Grid**: 0.2° tiles, reprojected to EPSG:32646 (UTM Zone 46N) at **20 m** resolution
- **Compositing**: 30 regular 12-day median bins per year, VV+VH dB, 1D linear temporal interpolation
- **Architecture**: Temporal 1D CNN with 3 independent binary heads (Aus, Aman, Boro)
- **Split**: Spatial tile-level split (80/10/10), never by pixel
- **Admin Units**: geoBoundaries ADM2 (districts) and ADM3 (upazilas)
- **Validation**: BBS Yearbook of Agricultural Statistics 2023 district totals

In [ ]:
# 1. Environment Setup & Dependency Installation
!pip install -q rasterio planetary-computer pystac-client torch torchvision shapely pyproj pandas scikit-learn tqdm

In [ ]:
# 2. System Hardware Verification
import multiprocessing as mp

import torch

print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
cpu_count = mp.cpu_count()
print(f"Available CPU cores: {cpu_count} (target Colab VM has 12 CPUs)")

In [ ]:
# 3. Clone Repository or Setup Paths
import sys
from pathlib import Path

repo_dir = Path("/content/bangladesh-rice-seasons")
if repo_dir.exists():
    sys.path.insert(0, str(repo_dir))
else:
    sys.path.insert(0, ".")

WORK_DIR = Path("/content/work")
COMP_DIR = WORK_DIR / "composites"
LABELS_DIR = WORK_DIR / "labels"
MODELS_DIR = WORK_DIR / "models"
PRED_DIR = WORK_DIR / "predictions"
AGG_DIR = WORK_DIR / "aggregates"
for d in [COMP_DIR, LABELS_DIR, MODELS_DIR, PRED_DIR, AGG_DIR]:
    d.mkdir(parents=True, exist_ok=True)
print(f"Workspace directories initialized at {WORK_DIR}")

In [ ]:
# 4. Generate/Load National 408 Tile Grid
from ricemap.tiles import generate_national_grid

tiles_json = WORK_DIR / "tiles_408.json"
tiles = generate_national_grid(out_json=tiles_json)
print(f"Loaded {len(tiles)} national grid tiles covering Bangladesh")

In [ ]:
# 5. National Batch S-1 Ingestion & Compositing across 12 CPUs (Resumable)
import time
from concurrent.futures import ProcessPoolExecutor, as_completed

from ricemap.composite import build_tile_timeseries

years = [2017, 2022, 2023]
tasks = [(tile, yr) for tile in tiles for yr in years]
print(f"Total tile-year composite tasks: {len(tasks)} (resumable: skips existing)")

def process_task(args):
    tile, yr = args
    return build_tile_timeseries(tile, yr, out_dir=COMP_DIR, max_workers=2, force=False)

t0 = time.time()
num_processes = min(6, mp.cpu_count())
completed = 0
with ProcessPoolExecutor(max_workers=num_processes) as executor:
    futures = {executor.submit(process_task, task): task for task in tasks}
    for fut in as_completed(futures):
        task = futures[fut]
        res = fut.result()
        completed += 1
        if completed % 10 == 0 or completed == len(tasks):
            elapsed = time.time() - t0
            print(f"Composited {completed}/{len(tasks)} tile-years ({elapsed/60.0:.1f} min elapsed)")

In [ ]:
# 6. Spatial Split & Training on Colab GPU
import numpy as np

from ricemap.model import (
    TemporalRiceCNN,
    compute_normalization_stats,
    get_auto_device,
    save_checkpoint,
    spatial_tile_split,
    train_model,
)

tile_ids = [t.tile_id for t in tiles]
splits = spatial_tile_split(tile_ids, train_ratio=0.8, val_ratio=0.1, test_ratio=0.1, seed=42)
print(f"Spatial split: {len(splits['train'])} train, {len(splits['val'])} val, {len(splits['test'])} test tiles")

train_data_list = []
train_lbl_list = []
for tid in splits["train"][:10]:
    c_p = COMP_DIR / f"{tid}_2017_timeseries.npz"
    l_p = LABELS_DIR / f"{tid}_labels_2017.npz"
    if c_p.exists() and l_p.exists():
        with np.load(c_p) as npz: train_data_list.append(npz["data"].astype(np.float32))
        with np.load(l_p) as npz: train_lbl_list.append(npz["labels"])

ch_means, ch_stds = compute_normalization_stats(train_data_list)
print(f"Float64 Normalization: VV mean={ch_means[0,0,0]:.2f}, VH mean={ch_means[0,0,1]:.2f}")

X_train = np.vstack([(d - ch_means) / ch_stds for d in train_data_list])
y_train = np.vstack(train_lbl_list)

device = get_auto_device()
model = TemporalRiceCNN()
res = train_model(
    model=model,
    train_data=X_train,
    train_labels=y_train,
    epochs=10,
    batch_size=16384,
    samples_per_epoch=500000,
    device=device
)
model_path = MODELS_DIR / "temporal_cnn_national.pt"
save_checkpoint(model, ch_means, ch_stds, "cnn", model_path)

In [ ]:
# 7. National Inference: 20m uint8 Bitmask Rasters
from ricemap.infer import run_tile_inference

for yr in [2017, 2022, 2023]:
    print(f"\n--- Running National Inference for Year {yr} ---")
    for t in tiles:
        c_p = COMP_DIR / f"{t.tile_id}_{yr}_timeseries.npz"
        if c_p.exists():
            run_tile_inference(model_path, c_p, out_dir=PRED_DIR, device=device)

In [ ]:
# 8. Administrative Aggregation (Districts ADM2 and Upazilas ADM3)
from ricemap.aggregate import aggregate_rice_area_by_admin

for yr in [2017, 2022, 2023]:
    year_tifs = sorted(PRED_DIR.glob(f"*_{yr}_rice_20m.tif"))
    if year_tifs:
        print(f"Aggregating {len(year_tifs)} tiles for year {yr}...")
        aggregate_rice_area_by_admin(year_tifs, "ADM2", yr, out_csv=AGG_DIR / f"district_area_{yr}.csv")
        aggregate_rice_area_by_admin(year_tifs, "ADM3", yr, out_csv=AGG_DIR / f"upazila_area_{yr}.csv")

In [ ]:
# 9. BBS Validation & Accuracy Reconciliation
from ricemap.validate import validate_district_predictions

bbs_csv = "/content/bangladesh-rice-seasons/data/bbs/district_rice_area.csv"
if Path(bbs_csv).exists():
    for yr in [2022, 2023]:
        d_csv = AGG_DIR / f"district_area_{yr}.csv"
        if d_csv.exists():
            validate_district_predictions(d_csv, bbs_csv, yr, out_csv=AGG_DIR / f"validation_{yr}.csv")
print("National pipeline execution complete!")